# E-Commerce Sales, Customer Segmentation & Purchase Prediction

This notebook covers:
1. Data Loading & Exploratory Data Analysis (EDA)
2. Data Preprocessing & Feature Engineering
3. Customer Segmentation using K-Means Clustering (RFM Analysis)
4. Purchase Prediction using Random Forest Classifier
5. Sales Forecasting using XGBoost Regressor
6. Model Evaluation & Visualization
7. Model Persistence (saving models for Flask API)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve,
                              mean_squared_error, r2_score, mean_absolute_error)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb
import joblib
import os

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('Libraries loaded successfully!')

## 2. Data Loading & Initial Exploration

In [ ]:
df = pd.read_csv('Ecommerce.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nMissing value percentage:')
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
print('Target variable distribution (purchased):')
print(df['purchased'].value_counts())
print('\nPurchase rate:', df['purchased'].mean().round(4))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('E-Commerce EDA - Key Distributions', fontsize=16, fontweight='bold')

# Purchase distribution
axes[0,0].pie(df['purchased'].value_counts(), labels=['Not Purchased','Purchased'],
              autopct='%1.1f%%', colors=['#FF6B6B','#4ECDC4'])
axes[0,0].set_title('Purchase Distribution')

# Revenue distribution
revenue_data = df[df['revenue'] > 0]['revenue']
axes[0,1].hist(revenue_data, bins=50, color='#45B7D1', edgecolor='white')
axes[0,1].set_title('Revenue Distribution (purchases only)')
axes[0,1].set_xlabel('Revenue')

# Device type
device_counts = df['device_type'].value_counts()
axes[0,2].bar(device_counts.index, device_counts.values, color=['#96CEB4','#FFEAA7','#DDA0DD'])
axes[0,2].set_title('Sessions by Device Type')
axes[0,2].set_xlabel('Device Type (0=Desktop, 1=Mobile, 2=Tablet)')

# Product category
cat_purchase = df.groupby('product_category')['purchased'].mean()
axes[1,0].bar(cat_purchase.index, cat_purchase.values, color='#F7DC6F')
axes[1,0].set_title('Purchase Rate by Product Category')
axes[1,0].set_xlabel('Category')
axes[1,0].set_ylabel('Purchase Rate')

# Time on site vs purchase
axes[1,1].boxplot([df[df['purchased']==0]['time_on_site_sec'],
                   df[df['purchased']==1]['time_on_site_sec']],
                  labels=['Not Purchased','Purchased'], patch_artist=True,
                  boxprops=dict(facecolor='#85C1E9', color='#2C3E50'))
axes[1,1].set_title('Time on Site vs Purchase')
axes[1,1].set_ylabel('Time (seconds)')

# Marketing channel
ch_purchase = df.groupby('marketing_channel')['purchased'].mean()
axes[1,2].bar(ch_purchase.index, ch_purchase.values, color='#F1948A')
axes[1,2].set_title('Purchase Rate by Marketing Channel')
axes[1,2].set_xlabel('Channel')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved as eda_overview.png')

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly sales trend
monthly = df[df['revenue']>0].groupby('visit_month')['revenue'].sum()
plt.figure(figsize=(12, 5))
plt.plot(monthly.index, monthly.values, marker='o', linewidth=2, color='#2196F3')
plt.fill_between(monthly.index, monthly.values, alpha=0.2, color='#2196F3')
plt.title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.xticks(range(1,13), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.savefig('monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing & Feature Engineering

In [ ]:
# Parse date
df['visit_date'] = pd.to_datetime(df['visit_date'], format='%d-%m-%Y')

# Drop columns not needed for modelling
drop_cols = ['session_id', 'visit_date', 'review_text', 'revenue_normalized']
df_model = df.drop(columns=drop_cols)

# Handle session_duration_bucket
duration_map = {'Very Short': 0, 'Short': 1, 'Medium': 2, 'Long': 3, 'Very Long': 4}
df_model['session_duration_bucket'] = df_model['session_duration_bucket'].map(duration_map)

print('Model dataframe shape:', df_model.shape)
df_model.head()

In [ ]:
# Additional features
df_model['price_per_item'] = df_model['unit_price'] * (1 - df_model['discount_percent']/100)
df_model['total_potential'] = df_model['unit_price'] * df_model['quantity']
df_model['cart_and_viewed'] = df_model['added_to_cart'] * df_model['pages_viewed']
df_model['is_weekend'] = df_model['visit_weekday'].apply(lambda x: 1 if x >= 5 else 0)

print('Engineered features added')
print(df_model.shape)

## 5. Customer Segmentation — K-Means Clustering (RFM)

In [ ]:
# Build RFM features per customer
df['visit_date'] = pd.to_datetime(df['visit_date'], errors='coerce')
snapshot_date = df['visit_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg(
    Recency=('visit_date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('session_id', 'count'),
    Monetary=('revenue', 'sum')
).reset_index()

print('RFM table shape:', rfm.shape)
rfm.describe()

In [ ]:
# Scale RFM features
scaler_rfm = StandardScaler()
rfm_scaled = scaler_rfm.fit_transform(rfm[['Recency','Frequency','Monetary']])

# Elbow method
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, marker='o', color='#E74C3C', linewidth=2)
plt.title('Elbow Method — Optimal K', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.tight_layout()
plt.savefig('elbow_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final clustering with K=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Visualize with PCA
pca = PCA(n_components=2, random_state=42)
rfm_pca = pca.fit_transform(rfm_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(rfm_pca[:,0], rfm_pca[:,1], c=rfm['Cluster'],
                      cmap='Set1', alpha=0.7, s=30)
plt.colorbar(scatter, label='Cluster')
plt.title('Customer Segments (PCA projection)', fontsize=13, fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.tight_layout()
plt.savefig('customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cluster profiles
cluster_profile = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(2)
print('Cluster Profiles:')
print(cluster_profile)

# Label clusters
cluster_labels = {}
for c in range(4):
    r = cluster_profile.loc[c, 'Recency']
    m = cluster_profile.loc[c, 'Monetary']
    if m > cluster_profile['Monetary'].mean() and r < cluster_profile['Recency'].mean():
        cluster_labels[c] = 'Champions'
    elif m > cluster_profile['Monetary'].mean():
        cluster_labels[c] = 'At-Risk High Value'
    elif r < cluster_profile['Recency'].mean():
        cluster_labels[c] = 'Recent Low Spend'
    else:
        cluster_labels[c] = 'Hibernating'

rfm['Segment'] = rfm['Cluster'].map(cluster_labels)
print('\nSegment distribution:')
print(rfm['Segment'].value_counts())

## 6. Purchase Prediction — Random Forest Classifier

In [ ]:
# Feature and target selection
feature_cols = [
    'device_type', 'user_type', 'marketing_channel', 'product_category',
    'unit_price', 'quantity', 'discount_percent', 'discount_amount',
    'pages_viewed', 'time_on_site_sec', 'added_to_cart', 'rating',
    'payment_method', 'visit_month', 'visit_weekday', 'visit_season',
    'session_duration_bucket', 'location', 'price_per_item',
    'total_potential', 'cart_and_viewed', 'is_weekend'
]

X = df_model[feature_cols]
y = df_model['purchased']

# Fill any remaining NaN
X = X.fillna(X.median())

print('Feature matrix shape:', X.shape)
print('Class balance:', y.value_counts().to_dict())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

In [ ]:
# Random Forest
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=12,
                                 min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_sc, y_train)
y_pred_rf = rf_clf.predict(X_test_sc)
y_prob_rf = rf_clf.predict_proba(X_test_sc)[:,1]

print('=== Random Forest Results ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob_rf):.4f}')
print(classification_report(y_test, y_pred_rf))

In [ ]:
# XGBoost
xgb_clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              use_label_encoder=False, eval_metric='logloss',
                              random_state=42)
xgb_clf.fit(X_train_sc, y_train)
y_pred_xgb = xgb_clf.predict(X_test_sc)
y_prob_xgb = xgb_clf.predict_proba(X_test_sc)[:,1]

print('=== XGBoost Results ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob_xgb):.4f}')
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# Confusion matrix & ROC curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Bought','Bought'], yticklabels=['Not Bought','Bought'])
axes[0].set_title('Confusion Matrix — Random Forest', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC curves
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
axes[1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, y_prob_rf):.3f})',
             color='#2196F3', linewidth=2)
axes[1].plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={roc_auc_score(y_test, y_prob_xgb):.3f})',
             color='#FF5722', linewidth=2)
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('ROC Curve Comparison', fontsize=12, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
fi = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(10, 8))
fi.tail(15).plot(kind='barh', color='#3F51B5')
plt.title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Revenue / Sales Forecasting — XGBoost Regressor

In [ ]:
# Only rows with actual purchases (revenue > 0)
df_rev = df_model[df_model['revenue'] > 0].copy()

X_rev = df_rev[feature_cols].fillna(df_rev[feature_cols].median())
y_rev = df_rev['revenue']

X_tr, X_te, y_tr, y_te = train_test_split(X_rev, y_rev, test_size=0.2, random_state=42)
scaler_rev = StandardScaler()
X_tr_sc = scaler_rev.fit_transform(X_tr)
X_te_sc = scaler_rev.transform(X_te)

xgb_reg = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_reg.fit(X_tr_sc, y_tr)
y_pred_rev = xgb_reg.predict(X_te_sc)

rmse = np.sqrt(mean_squared_error(y_te, y_pred_rev))
mae = mean_absolute_error(y_te, y_pred_rev)
r2 = r2_score(y_te, y_pred_rev)
print(f'Revenue Regression — RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}')

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_te, y_pred_rev, alpha=0.3, color='#9C27B0', s=15)
max_val = max(y_te.max(), y_pred_rev.max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Fit')
plt.title(f'Actual vs Predicted Revenue  (R²={r2:.3f})', fontsize=13, fontweight='bold')
plt.xlabel('Actual Revenue')
plt.ylabel('Predicted Revenue')
plt.legend()
plt.tight_layout()
plt.savefig('revenue_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Models & Artifacts

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(rf_clf,       'models/purchase_classifier.pkl')
joblib.dump(xgb_reg,      'models/revenue_regressor.pkl')
joblib.dump(kmeans,       'models/customer_segmentation.pkl')
joblib.dump(scaler,       'models/feature_scaler.pkl')
joblib.dump(scaler_rev,   'models/revenue_scaler.pkl')
joblib.dump(scaler_rfm,   'models/rfm_scaler.pkl')
joblib.dump(feature_cols, 'models/feature_cols.pkl')

import json
with open('models/cluster_labels.json', 'w') as f:
    json.dump({str(k): v for k, v in cluster_labels.items()}, f)

print('All models saved to /models folder:')
for f in os.listdir('models'):
    print(' ', f)

## 9. Model Summary

In [ ]:
summary = pd.DataFrame({
    'Model': ['Random Forest Classifier', 'XGBoost Classifier',
              'XGBoost Regressor', 'K-Means Clustering'],
    'Task': ['Purchase Prediction', 'Purchase Prediction',
             'Revenue Forecasting', 'Customer Segmentation'],
    'Metric': ['Accuracy / ROC-AUC', 'Accuracy / ROC-AUC', 'RMSE / R²', 'Inertia'],
    'Score': [
        f'{accuracy_score(y_test, y_pred_rf):.4f} / {roc_auc_score(y_test, y_prob_rf):.4f}',
        f'{accuracy_score(y_test, y_pred_xgb):.4f} / {roc_auc_score(y_test, y_prob_xgb):.4f}',
        f'{rmse:.2f} / {r2:.4f}',
        f'{kmeans.inertia_:.2f}'
    ]
})
print(summary.to_string(index=False))